In [ ]:
from pathlib import Path
import pandas as pd

import ixmp4
import pyam
import nomenclature

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
ar6_db = pyam.iiasa.Connection("ar6-public")

In [ ]:
ssp_models = [
#    'AIM/CGE 2.0',
#    'GCAM 4.2',
    'IMAGE 3.0.1',
#    'MESSAGE-GLOBIOM 1.0',
#    'REMIND-MAgPIE 1.5',
#    'WITCH-GLOBIOM 3.1'
]

In [ ]:
ssp_scenarios = [
    'SSP1-19',
    'SSP1-26',
    'SSP1-34',
    'SSP1-45',
    'SSP1-Baseline',
    'SSP2-19',
    'SSP2-26',
    'SSP2-34',
    'SSP2-45',
    'SSP2-60',
    'SSP2-Baseline',
    'SSP3-34',
    'SSP3-45',
    'SSP3-60',
    'SSP3-Baseline',
    'SSP4-19',
    'SSP4-26',
    'SSP4-34',
    'SSP4-45',
    'SSP4-60',
    'SSP4-Baseline',
    'SSP5-19',
    'SSP5-26',
    'SSP5-34',
    'SSP5-45',
    'SSP5-60',
    'SSP5-Baseline',
]

In [ ]:
for model in ssp_models:
    df = ar6_db.query(model=model, scenario=ssp_scenarios)
    if not df.empty:
        df.to_excel(f"raw/AR6/SSP_{model.replace(" ", "_").replace("/", "_")}.xlsx")

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/AR6/").iterdir()) if i.name.startswith("SSP_")
    ]
)

In [ ]:
meta = pd.read_excel("raw/AR6/AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx", sheet_name="meta_Ch3vetted_withclimate")

In [ ]:
meta = meta[['Model', 'Scenario', 'Literature Reference (if applicable)', 'Category_subset']]
meta.rename(
    columns={
        "Model": "model",
        "Scenario": "scenario",
        "Category_subset": "Climate Assessment|AR6|Category [ID]",
        "Literature Reference (if applicable)": "manuscript",
    },
    inplace=True,
)

In [ ]:
df = pyam.IamDataFrame(df.data, meta=meta)

In [ ]:
df.meta["Climate Assessment|AR6|Category [ID]"].unique()

In [ ]:
dict()

In [ ]:
df.meta.replace(
    {
        'C1a_NZGHGs': "C1a",
        'C3y_+veGHGs': "C3y",
        'C3x_NZGHGs': "C3x",
    },
    inplace=True,
)

In [ ]:
df_no_assessment = df.filter(**{"Climate Assessment|AR6|Category [ID]": "C*"}, keep=False)
df_no_assessment

In [ ]:
df_no_assessment.meta

In [ ]:
df_assessed = df.filter(**{"Climate Assessment|AR6|Category [ID]": "C*"})

In [ ]:
df_assessed.meta

In [ ]:
runs = platform.runs.tabulate(scenario="SSP*")

In [ ]:
pending_index = df.index.difference(runs.set_index(["model", "scenario"]).index)
pending_index

In [ ]:
df.filter(index=pending_index, inplace=True)

In [ ]:
df.meta

In [ ]:
for is_19, doi, name in [
    (False, "10.1016/j.gloenvcha.2016.05.009", "Riahi et al. (2017)"),
    (True, "10.1038/s41558-018-0091-3", "Rogelj et al. (2018)"),
]:
    index = df.filter(scenario="SSP*-19", keep=is_19).index
    df.set_meta(meta=doi, name="Scientific Manuscript (DOI)", index=index)
    df.set_meta(meta=name, name="Scientific Manuscript (Citation)", index=index)


In [ ]:
df.meta.drop(columns="manuscript", inplace=True)

In [ ]:
df.meta.loc['WITCH-GLOBIOM 3.1']

In [ ]:
df.region

In [ ]:
region_mapping = {
  "Asian countries except Japan": "Asia (R5)",
  "Latin American countries": "Latin America (R5)",
  "Countries of the Middle East and Africa": "Middle East & Africa (R5)",
  "OECD90 and EU (and EU candidate) countries": "OECD & EU (R5)",
  "Countries from the Reforming Economies of the Former Soviet Union": "Reforming Economies (R5)",
}

In [ ]:
df = df.rename(region=region_mapping)

In [ ]:
df

In [ ]:
df.region

In [ ]:
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
        "Diagnostics|MAGICC6*",
        "Carbon Sequestration|Other",
        "Secondary Energy",
        "Food Energy Supply",
        "Investment|Energy Supply|Electricity|Non-fossil",
        "Investment|Energy Supply|Extraction|Bioenergy",
        "Investment|Energy Supply|Hydrogen|Renewable",
        "Policy Cost|Consumption Loss",
        "Policy Cost|GDP Loss",
        "Policy Cost|Area under MAC Curve",
        "Price|Agriculture|Non-Energy Crops and Livestock|Index",
        "Policy Cost|Additional Total Energy System Cost",
        "Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry", 
        "Final Energy|Residential and Commercial|Solids|Biomass|Traditional",
        "Final Energy|Transportation|Liquids|Natural Gas",
        "Capacity Additions|Electricity|Storage Capacity",
        "Capacity|Electricity|Peak Demand",
        "Capacity|Electricity|Storage",
        "Secondary Energy|Electricity|Curtailment",
        "Secondary Energy|Electricity|Curtailment|Solar",
        "Secondary Energy|Electricity|Curtailment|Wind",
        "Secondary Energy|Electricity|Storage",
        "Secondary Energy|Electricity|Storage Losses",
        "Secondary Energy|Electricity|Transmission Losses",
    ],
    keep=False,
    inplace=True
)

In [ ]:
# update carbon-management variables
carbon_management_mapping = {
    "Agricultural Demand|Crops|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Crops|Energy|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Crops|Energy|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
}

df.rename(variable=carbon_management_mapping, inplace=True)

In [ ]:
project = "engage"
legacy_mapping = {}

for code, attrs in definition.variable.items():
    if project in attrs.extra_attributes:
        legacy_mapping[attrs.__getattr__(project)] = code

df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "million m3/yr": "km3/yr",
    },
    inplace=True,
)

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
df.filter(region=["World", "*(R5)"], inplace=True)

In [ ]:
definition.validate(df)

In [ ]:
df.set_meta("SSP", "Project")

In [ ]:
df.meta

In [ ]:
for model in df.model:
    df.filter(model=model).to_ixmp4(platform)
    print(model)

In [ ]:
df